In [60]:
import random, math
import numpy as np
from tslearn.metrics import dtw, dtw_path
from timeit import default_timer as timer

In [61]:
# Setting up the inputs
start = timer()

inputSize = 20
Alist = []
for i in range(inputSize):
#   randomx.append(random.randint(1,1000))
    Alist.append(i)
A = np.array(Alist)
print(A)

Blist = []
for i in range(inputSize, inputSize*2):
    #randomy.append(random.randint(1,1000))
    Blist.append(i)
B = np.array(Blist)
print(B)

print("DONE GENERATING")

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]
[20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39]
DONE GENERATING


In [62]:
#Base DTW 
#dtw_score = dtw(x, y)
# Or, if the path is also an important information:
#optimal_path, dtw_score = dtw_path(x, y)

In [63]:
# Modified Dynamic Time Warping Start

In [64]:
# Generate the subsequences given a size g and the Sequence to be split
def generateSubsequences(subsequenceSize, sequenceInput):
    numberOfSubsequences = math.floor(len(sequenceInput)/(subsequenceSize-1))

    print(len(sequenceInput), numberOfSubsequences)

    allSubsequences = []

    for i in range(numberOfSubsequences):
        startIndex = i*(subsequenceSize-1)
        newSubsequence = []
        if i > 0:
            newSubsequence.append(allSubsequences[i-1][-1])
        for j in range(subsequenceSize-1):
            newSubsequence.append(sequenceInput[startIndex])
            startIndex += 1
        allSubsequences.append(np.array(newSubsequence))

    #print(Asequences)
    return allSubsequences

# Subsequence size of 5 for sequences A and B
subsequenceSize = 5
subsequencesA = generateSubsequences(subsequenceSize, A)
subsequencesB = generateSubsequences(subsequenceSize, B)

20 5
20 5


In [65]:
# Compute the cost matrix given two sequences (or subsequences)
def computeDistanceMatrix(A, B, subsequenceSize):
    distanceMatrix = np.zeros((subsequenceSize, subsequenceSize))

    # Add a zero to the start of the sequence if it is smaller than the subsequence size. 
    # This would be for the first subsequence only.
    if len(A) < subsequenceSize:
        A = np.insert(A, 0, 0)
    if len(B) < subsequenceSize:
        B = np.insert(B, 0, 0)

    # Calculate distance matrix. 
    # I feel like there is a better way to implement this than 2 FOR loops.
    for i in range(subsequenceSize):
        for j in range(subsequenceSize):
            distanceMatrix[i, j] = abs(A[i] - B[j])
    
    return distanceMatrix

    # Calculating Cost Matrix. Do not need this yet but left it here for future use.
    #costMatrix = np.zeros((subsequenceSize+1, subsequenceSize+1))
    #costMatrix[1:, 0] = np.inf
    #costMatrix[0, 1:] = np.inf
    #print(costMatrix)

    #for i in range(1, subsequenceSize+1):
    #    for j in range(1, subsequenceSize+1):
    #        costMatrix[i,j] = distanceMatrix[i-1, j-1] + min(costMatrix[i-1, j-1], 
    #                                                            costMatrix[i-1,j], costMatrix[i, j-1])
    #print(costMatrix)
    
#print(subsequencesA[0], subsequencesB[0])
#computeDistanceMatrix(subsequencesA[1], subsequencesB[1], subsequenceSize)
#computeDistanceMatrix(subsequencesA[0], subsequencesB[0], subsequenceSize)

In [66]:
# Calculate all distance matrices with the the indices of the subsequences as a tuple key.

distanceMatrices = {}
for i in range(subsequenceSize):
    for j in range(subsequenceSize):
        key = (i, j)
        value = computeDistanceMatrix(subsequencesA[i], subsequencesB[j], subsequenceSize)
        distanceMatrices[key] = value
        #print(i, j, distanceMatrices[(i,j)])
        
print(len(distanceMatrices), distanceMatrices[(1,4)])
print(distanceMatrices[(1,4)][1,0])
print(distanceMatrices[(1,4)][0,4])
print(distanceMatrices[(1,4)][0:,0])

25 [[32. 33. 34. 35. 36.]
 [31. 32. 33. 34. 35.]
 [30. 31. 32. 33. 34.]
 [29. 30. 31. 32. 33.]
 [28. 29. 30. 31. 32.]]
31.0
36.0
[32. 31. 30. 29. 28.]


In [67]:
# Keys for L and R
horizontalL = []
verticalL = []

horizontalR = []
verticalR = []
for i in range(subsequenceSize-1, -1, -1):
    # Indices for L
    indexHorL = (0, i)
    horizontalL.append(indexHorL)
    
    # Indices for R
    indexHorR = (subsequenceSize-1, i)
    horizontalR.append(indexHorR)

for i in range(subsequenceSize):
    # Indices for L
    indexVerL = (i, 0)
    verticalL.append(indexVerL)

    # Indices for R
    indexVerR = (i, subsequenceSize-1)
    verticalR.append(indexVerR)

# the pop removes the duplicate entry where the horizontal and vertical touch
horizontalL.pop()
indicesL = tuple(horizontalL + verticalL)

verticalR.pop()
indicesR = tuple(verticalR + horizontalR)

print("L : ", indicesL)
#for i in indicesL:
#    print(distanceMatrices[(1,4)][i[0], i[1]])

print("R : ", indicesR)
#for i in keysR:
#    print(distanceMatrices[(1,4)][i[0], i[1]])
    

L :  ((0, 4), (0, 3), (0, 2), (0, 1), (0, 0), (1, 0), (2, 0), (3, 0), (4, 0))
R :  ((0, 4), (1, 4), (2, 4), (3, 4), (4, 4), (4, 3), (4, 2), (4, 1), (4, 0))


In [68]:
# Enumerate all admissible pairs of positions

pairsOfPositions = []
for i in indicesL:
    for j in indicesR:
        if i != j and i[0] <= j[0] and i[1] <= j[1]:
            vw = (i, j)
            pairsOfPositions.append(vw)

print(len(pairsOfPositions), pairsOfPositions)

59 [((0, 4), (1, 4)), ((0, 4), (2, 4)), ((0, 4), (3, 4)), ((0, 4), (4, 4)), ((0, 3), (0, 4)), ((0, 3), (1, 4)), ((0, 3), (2, 4)), ((0, 3), (3, 4)), ((0, 3), (4, 4)), ((0, 3), (4, 3)), ((0, 2), (0, 4)), ((0, 2), (1, 4)), ((0, 2), (2, 4)), ((0, 2), (3, 4)), ((0, 2), (4, 4)), ((0, 2), (4, 3)), ((0, 2), (4, 2)), ((0, 1), (0, 4)), ((0, 1), (1, 4)), ((0, 1), (2, 4)), ((0, 1), (3, 4)), ((0, 1), (4, 4)), ((0, 1), (4, 3)), ((0, 1), (4, 2)), ((0, 1), (4, 1)), ((0, 0), (0, 4)), ((0, 0), (1, 4)), ((0, 0), (2, 4)), ((0, 0), (3, 4)), ((0, 0), (4, 4)), ((0, 0), (4, 3)), ((0, 0), (4, 2)), ((0, 0), (4, 1)), ((0, 0), (4, 0)), ((1, 0), (1, 4)), ((1, 0), (2, 4)), ((1, 0), (3, 4)), ((1, 0), (4, 4)), ((1, 0), (4, 3)), ((1, 0), (4, 2)), ((1, 0), (4, 1)), ((1, 0), (4, 0)), ((2, 0), (2, 4)), ((2, 0), (3, 4)), ((2, 0), (4, 4)), ((2, 0), (4, 3)), ((2, 0), (4, 2)), ((2, 0), (4, 1)), ((2, 0), (4, 0)), ((3, 0), (3, 4)), ((3, 0), (4, 4)), ((3, 0), (4, 3)), ((3, 0), (4, 2)), ((3, 0), (4, 1)), ((3, 0), (4, 0)), ((4, 0

In [69]:
# Checks for all absolute possible moves for the node (ignoring pair of position restrictions)
def checkNextNodes(currentNode):
    nextNodes = []
    
    # Check for Horizontal Move
    if currentNode[1] < subsequenceSize-1:
        nextNodes.append((currentNode[0], currentNode[1]+1))
    # Check for Vertical Move
    if currentNode[0] < subsequenceSize-1:
        nextNodes.append((currentNode[0]+1, currentNode[1]))
    # Check for Diagonal Move, diagonal move is only possible if vertical and horizontal moves are possible
    if len(nextNodes) == 2:
        nextNodes.append((currentNode[0]+1, currentNode[1]+1))

    return nextNodes

# Get all possible movements from the nodes into a dictionary
nodeMoves = {}
for i in range(subsequenceSize):
    for j in range(subsequenceSize):
        currentNode = (i,j)
        nextNodes = checkNextNodes(currentNode)
        nodeMoves[currentNode] = nextNodes
        
print(len(nodeMoves))
#for key, value in nodeMoves.items():
#    print(f"{key}: {value}")

print(nodeMoves[(0,0)])

25
[(0, 1), (1, 0), (1, 1)]


In [70]:
# Enumerate all paths 

pathsDict = {}
# testing: L: (0,3) and R: (3,4) (7 paths)
for pair in pairsOfPositions:
#for pair in [((0, 4), (2, 4))]:
    print("CURRENT PAIR: ", pair)
    currentNode = pair[0]
    endNode = pair[1]
    allPaths = []
    unfinishedPaths = []

    # Check starting node's moves if they are legal
    for move in nodeMoves[currentNode]:
        #print(move)
        if move[0] <= endNode[0] and move[1] <= endNode[1]:
            unfinishedPaths.append([move])
        if move == endNode:
            allPaths.append(move)
            unfinishedPaths.remove([move])
    #print(f"Unfinished paths: {unfinishedPaths}")
    #print(f"Completed paths: {allPaths}")
            
    while len(unfinishedPaths) > 0:
        # print(unfinishedPaths)
        currentPath = unfinishedPaths[0]
    
        raw_moves = nodeMoves[currentPath[-1]]
        moves = [x for x in raw_moves if x[0] <= endNode[0] and x[1] <= endNode[1]]
        # If there are more than 1 possible moves, make copies of the current path and append the moves
        while len(moves) > 1:
            pathCopy = currentPath.copy()
            pathCopy.append(moves[0])
            unfinishedPaths.append(pathCopy)
            moves.pop(0)
        currentPath.append(moves[0])

        # If the move is the end node already, put the path in allPaths to mark it completed
        if moves[0] == endNode:
            #print("POP IT")
            allPaths.append(currentPath)
            unfinishedPaths.pop(0)
            
        #print(f'Current: {currentPath}')
        #print(f"Unfinished: {unfinishedPaths}")
    pathsDict[pair] = allPaths
        
for key, value in pathsDict.items():
    print(f"{key}: {len(value)}")
    #print(f"{key}: {value}")
print(pathsDict[((0,3), (2,4))])

CURRENT PAIR:  ((0, 4), (1, 4))
CURRENT PAIR:  ((0, 4), (2, 4))
CURRENT PAIR:  ((0, 4), (3, 4))
CURRENT PAIR:  ((0, 4), (4, 4))
CURRENT PAIR:  ((0, 3), (0, 4))
CURRENT PAIR:  ((0, 3), (1, 4))
CURRENT PAIR:  ((0, 3), (2, 4))
CURRENT PAIR:  ((0, 3), (3, 4))
CURRENT PAIR:  ((0, 3), (4, 4))
CURRENT PAIR:  ((0, 3), (4, 3))
CURRENT PAIR:  ((0, 2), (0, 4))
CURRENT PAIR:  ((0, 2), (1, 4))
CURRENT PAIR:  ((0, 2), (2, 4))
CURRENT PAIR:  ((0, 2), (3, 4))
CURRENT PAIR:  ((0, 2), (4, 4))
CURRENT PAIR:  ((0, 2), (4, 3))
CURRENT PAIR:  ((0, 2), (4, 2))
CURRENT PAIR:  ((0, 1), (0, 4))
CURRENT PAIR:  ((0, 1), (1, 4))
CURRENT PAIR:  ((0, 1), (2, 4))
CURRENT PAIR:  ((0, 1), (3, 4))
CURRENT PAIR:  ((0, 1), (4, 4))
CURRENT PAIR:  ((0, 1), (4, 3))
CURRENT PAIR:  ((0, 1), (4, 2))
CURRENT PAIR:  ((0, 1), (4, 1))
CURRENT PAIR:  ((0, 0), (0, 4))
CURRENT PAIR:  ((0, 0), (1, 4))
CURRENT PAIR:  ((0, 0), (2, 4))
CURRENT PAIR:  ((0, 0), (3, 4))
CURRENT PAIR:  ((0, 0), (4, 4))
CURRENT PAIR:  ((0, 0), (4, 3))
CURRENT 

In [ ]:
print(dtw_score)
print(optimal_path)

end = timer()
print(end-start)